# Building an Aligned Multi-Tool Research Agent
## Part 2: Trajectories and Curriculum Learning

**Learning Objectives:**
- Implement trajectory-level alignment constraints
- Build curriculum learning system with 4 progressive stages
- Understand transfer learning between complexity levels
- Apply mathematical advancement criteria for stage progression

**What You'll Build:**
A complete curriculum learning system that progressively trains agents from simple single-tool decisions to complex multi-step aligned reasoning.

**Prerequisites:**
- Completion of Part 1: Mathematical Foundations
- Understanding of trajectory-based reinforcement learning
- Familiarity with module3.md curriculum learning framework

## Setup and Dependencies

**Note:** This notebook builds on Part 1. Make sure you've run Part 1 first or copy the core classes below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Union
from dataclasses import dataclass
from enum import Enum
import random
from collections import defaultdict
import json
import time
from datetime import datetime

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

# Configure plotting
try:
    plt.style.use('seaborn-v0_8')
except:
    plt.style.use('seaborn')  # Fallback for older versions

sns.set_palette("husl")

print("Dependencies loaded successfully!")
print("Ready to build trajectory and curriculum learning systems.")

## Import Core Classes from Part 1

If you haven't run Part 1, you'll need to define these core classes. Otherwise, you can import them.

In [ ]:
# Core classes from Part 1 - copy these if Part 1 isn't available

@dataclass
class ProblemState:
    """Problem State: s_problem = [query_type, complexity_level, domain, stakeholders]"""
    query_type: str  # 'factual', 'analytical', 'controversial', 'urgent'
    complexity_level: float  # 0.0 to 1.0
    domain: str  # 'science', 'politics', 'technology', 'general'
    stakeholders: List[str]  # affected parties
    
    def to_vector(self) -> np.ndarray:
        """Convert to numerical representation for RL algorithms"""
        query_encoding = {'factual': 0, 'analytical': 1, 'controversial': 2, 'urgent': 3}
        domain_encoding = {'science': 0, 'politics': 1, 'technology': 2, 'general': 3}
        
        return np.array([
            query_encoding.get(self.query_type, 0),
            self.complexity_level,
            domain_encoding.get(self.domain, 0),
            len(self.stakeholders)
        ])

@dataclass 
class ContextState:
    """Context State: s_context = [time_pressure, quality_requirements, user_expertise, urgency_level]"""
    time_pressure: float  # 0.0 to 1.0
    quality_requirements: float  # 0.0 to 1.0  
    user_expertise: float  # 0.0 to 1.0
    urgency_level: float  # 0.0 to 1.0
    
    def to_vector(self) -> np.ndarray:
        return np.array([self.time_pressure, self.quality_requirements, 
                        self.user_expertise, self.urgency_level])

@dataclass
class ResourceState:
    """Resource State: s_resources = [budget_remaining, time_remaining, tool_availability, api_limits]"""
    budget_remaining: float  # 0.0 to 1.0 (normalized)
    time_remaining: float   # 0.0 to 1.0 (normalized)
    tool_availability: Dict[str, bool]  # which tools are available
    api_limits: Dict[str, float]  # remaining API calls per tool
    
    def to_vector(self) -> np.ndarray:
        available_tools = sum(self.tool_availability.values())
        avg_api_remaining = np.mean(list(self.api_limits.values()))
        return np.array([self.budget_remaining, self.time_remaining, 
                        available_tools/12, avg_api_remaining])

@dataclass
class ConstraintState:
    """Constraint State: s_constraints = [privacy_level, compliance_requirements, user_values, safety_thresholds]"""
    privacy_level: float  # 0.0 to 1.0
    compliance_requirements: List[str]  # regulatory constraints
    user_values: Dict[str, float]  # accuracy, speed, cost, safety weights
    safety_thresholds: Dict[str, float]  # minimum safety requirements
    
    def to_vector(self) -> np.ndarray:
        compliance_score = len(self.compliance_requirements) / 5  # normalize
        values_vector = [self.user_values.get(k, 0.5) for k in ['accuracy', 'speed', 'cost', 'safety']]
        safety_score = np.mean(list(self.safety_thresholds.values()))
        return np.array([self.privacy_level, compliance_score, *values_vector, safety_score])

class AgentState:
    """Complete agent state combining all components"""
    def __init__(self, problem: ProblemState, context: ContextState, 
                 resources: ResourceState, constraints: ConstraintState,
                 history: List[Tuple] = None):
        self.problem = problem
        self.context = context 
        self.resources = resources
        self.constraints = constraints
        self.history = history or []  # [(action, reward, outcome), ...]
    
    def to_vector(self) -> np.ndarray:
        """Convert complete state to vector for RL algorithms"""
        # History encoding: last 3 actions and their outcomes
        history_vector = np.zeros(9)  # 3 * (action_id + reward + outcome_quality)
        for i, (action_id, reward, outcome_quality) in enumerate(self.history[-3:]):
            base_idx = i * 3
            history_vector[base_idx:base_idx+3] = [action_id, reward, outcome_quality]
            
        return np.concatenate([
            self.problem.to_vector(),
            self.context.to_vector(), 
            self.resources.to_vector(),
            self.constraints.to_vector(),
            history_vector
        ])
    
    def is_alignment_complete(self) -> bool:
        """Check if state contains sufficient information for aligned decisions"""
        has_user_values = len(self.constraints.user_values) >= 4
        has_safety_thresholds = len(self.constraints.safety_thresholds) >= 2
        has_context = self.context.quality_requirements > 0
        
        return has_user_values and has_safety_thresholds and has_context

class ResearchTool(Enum):
    """Enumeration of 12 research tools with unique properties"""
    ACADEMIC_SEARCH = 0
    WEB_SEARCH = 1
    NEWS_SEARCH = 2
    FACT_CHECK = 3
    SENTIMENT_ANALYSIS = 4
    CITATION_ANALYSIS = 5
    SUMMARIZATION = 6
    CROSS_REFERENCE = 7
    BIAS_DETECTION = 8
    CONFIDENCE_ASSESSMENT = 9
    HUMAN_CONSULTATION = 10
    SYNTHESIS = 11

@dataclass
class ToolProperties:
    """Tool feature vector: f(a_i) = [c_i, t_i, p_i, s_i]"""
    cost: float          # computational cost
    time: float          # time requirement
    reliability: float   # reliability score
    accuracy: float      # accuracy strength
    speed: float         # speed strength
    coverage: float      # information coverage
    safety: float        # safety level
    
    def feature_vector(self) -> np.ndarray:
        return np.array([self.cost, self.time, self.reliability, 
                        self.accuracy, self.speed, self.coverage, self.safety])

# Simple action space for this notebook
class MultiToolActionSpace:
    def __init__(self):
        self.tool_properties = {
            ResearchTool.ACADEMIC_SEARCH: ToolProperties(0.7, 0.8, 0.95, 0.95, 0.2, 0.6, 0.9),
            ResearchTool.WEB_SEARCH: ToolProperties(0.2, 0.1, 0.6, 0.6, 0.95, 0.9, 0.5),
            ResearchTool.FACT_CHECK: ToolProperties(0.8, 0.6, 0.9, 0.9, 0.4, 0.3, 0.95),
            ResearchTool.BIAS_DETECTION: ToolProperties(0.7, 0.6, 0.85, 0.85, 0.4, 0.3, 0.95),
            ResearchTool.HUMAN_CONSULTATION: ToolProperties(1.0, 1.0, 0.95, 0.9, 0.1, 0.6, 1.0),
            ResearchTool.SYNTHESIS: ToolProperties(0.8, 0.9, 0.8, 0.85, 0.2, 0.95, 0.8),
            # Add simplified properties for other tools
            ResearchTool.NEWS_SEARCH: ToolProperties(0.3, 0.2, 0.7, 0.7, 0.8, 0.7, 0.6),
            ResearchTool.SENTIMENT_ANALYSIS: ToolProperties(0.4, 0.3, 0.8, 0.8, 0.7, 0.4, 0.8),
            ResearchTool.CITATION_ANALYSIS: ToolProperties(0.6, 0.5, 0.85, 0.85, 0.5, 0.5, 0.9),
            ResearchTool.SUMMARIZATION: ToolProperties(0.3, 0.2, 0.75, 0.75, 0.8, 0.8, 0.7),
            ResearchTool.CROSS_REFERENCE: ToolProperties(0.9, 0.7, 0.9, 0.9, 0.3, 0.9, 0.85),
            ResearchTool.CONFIDENCE_ASSESSMENT: ToolProperties(0.5, 0.4, 0.8, 0.8, 0.6, 0.5, 0.9)
        }
    
    def get_tool_properties(self, tool: ResearchTool) -> ToolProperties:
        return self.tool_properties[tool]
    
    def get_action_count(self) -> int:
        return len(ResearchTool)

# Simple reward matrix for this notebook
class StochasticRewardMatrix:
    def __init__(self, action_space):
        self.action_space = action_space
    
    def get_stochastic_reward(self, state: AgentState, action: ResearchTool) -> float:
        # Simplified reward calculation
        props = self.action_space.get_tool_properties(action)
        base_reward = props.accuracy * 5 + props.speed * 3 + props.safety * 4
        return base_reward + np.random.normal(0, 1.0)
    
    def get_alignment_reward(self, state: AgentState, action: ResearchTool) -> float:
        props = self.action_space.get_tool_properties(action)
        user_values = state.constraints.user_values
        return (
            user_values.get('accuracy', 0.5) * props.accuracy +
            user_values.get('speed', 0.5) * props.speed +
            user_values.get('safety', 0.5) * props.safety
        ) * 2.0

# Initialize core components
action_space = MultiToolActionSpace()
reward_matrix = StochasticRewardMatrix(action_space)

print("Core classes loaded successfully!")
print(f"Action space size: {action_space.get_action_count()}")

## Part 1: Trajectory System with Alignment Constraints

Implementing trajectory-level alignment following the mathematical framework:
$$\tau = (s_0, a_0, r_0, s_1, a_1, r_1, ..., s_T, a_T, r_T)$$

With alignment constraints throughout the trajectory, not just at the end.

In [ ]:
@dataclass
class TrajectoryStep:
    """Single step in trajectory: (state, action, reward, next_state, info)"""
    state: AgentState
    action: ResearchTool
    reward: float
    next_state: AgentState
    alignment_score: float
    outcome_quality: float
    info: Dict

class AlignedTrajectory:
    """Trajectory with alignment constraints and evaluation"""
    
    def __init__(self, max_length: int = 10):
        self.steps: List[TrajectoryStep] = []
        self.max_length = max_length
        self.user_values = None
        
    def add_step(self, step: TrajectoryStep):
        """Add step with alignment validation"""
        self.steps.append(step)
        
    def get_trajectory_length(self) -> int:
        return len(self.steps)
    
    def compute_alignment_score(self) -> float:
        """Compute cumulative alignment score: A_cumulative(τ) = Σ α^t · alignment_score(s_t, a_t)"""
        if not self.steps:
            return 0.0
            
        alpha = 0.95  # Discount factor for recent alignment
        cumulative_score = 0.0
        
        for t, step in enumerate(self.steps):
            weight = alpha ** t
            cumulative_score += weight * step.alignment_score
            
        return cumulative_score
    
    def check_consistency_constraint(self) -> bool:
        """Check if trajectory maintains value consistency across time"""
        if len(self.steps) < 2:
            return True
            
        # Check that similar states lead to consistent value-aligned actions
        for i in range(len(self.steps) - 1):
            for j in range(i + 1, len(self.steps)):
                state_i = self.steps[i].state
                state_j = self.steps[j].state
                action_i = self.steps[i].action
                action_j = self.steps[j].action
                
                # Check state similarity (simplified)
                if self._states_similar(state_i, state_j):
                    if not self._actions_value_consistent(action_i, action_j, state_i):
                        return False
        return True
    
    def _states_similar(self, s1: AgentState, s2: AgentState, threshold: float = 0.3) -> bool:
        """Check if two states are similar enough to expect consistent actions"""
        # Compare problem types and contexts
        same_query_type = s1.problem.query_type == s2.problem.query_type
        similar_complexity = abs(s1.problem.complexity_level - s2.problem.complexity_level) < threshold
        similar_urgency = abs(s1.context.urgency_level - s2.context.urgency_level) < threshold
        
        return same_query_type and similar_complexity and similar_urgency
    
    def _actions_value_consistent(self, a1: ResearchTool, a2: ResearchTool, state: AgentState) -> bool:
        """Check if two actions are consistent with user values"""
        # Actions are consistent if they prioritize the same top user value
        user_values = state.constraints.user_values
        top_value = max(user_values, key=user_values.get)
        
        # Get tool properties
        props1 = action_space.get_tool_properties(a1)
        props2 = action_space.get_tool_properties(a2)
        
        # Check if both actions align with top user value
        if top_value == 'accuracy':
            return props1.accuracy > 0.7 and props2.accuracy > 0.7
        elif top_value == 'speed':
            return props1.speed > 0.7 and props2.speed > 0.7
        elif top_value == 'safety':
            return props1.safety > 0.8 and props2.safety > 0.8
        else:
            return True  # Default to consistent
    
    def check_progressive_refinement(self, min_threshold: float = 0.1) -> bool:
        """Check if trajectory shows progressive information quality improvement"""
        if len(self.steps) < 2:
            return True
            
        quality_improvements = 0
        total_steps = len(self.steps) - 1
        
        for i in range(1, len(self.steps)):
            current_quality = self.steps[i].outcome_quality
            previous_quality = self.steps[i-1].outcome_quality
            
            if current_quality >= previous_quality:
                quality_improvements += 1
            elif self._justified_exploration(self.steps[i]):
                quality_improvements += 0.5  # Partial credit for justified exploration
                
        improvement_ratio = quality_improvements / total_steps
        return improvement_ratio >= min_threshold
    
    def _justified_exploration(self, step: TrajectoryStep) -> bool:
        """Check if a step represents justified exploration (lower quality but strategic)"""
        # Exploration is justified if it's a bias detection, fact check, or cross-reference tool
        # even if immediate quality is lower
        exploration_tools = {ResearchTool.BIAS_DETECTION, ResearchTool.FACT_CHECK, 
                           ResearchTool.CROSS_REFERENCE, ResearchTool.CONFIDENCE_ASSESSMENT}
        return step.action in exploration_tools
    
    def check_resource_rationality(self) -> bool:
        """Check if trajectory uses resources rationally"""
        if not self.steps:
            return True
            
        initial_budget = self.steps[0].state.resources.budget_remaining
        final_budget = self.steps[-1].next_state.resources.budget_remaining
        budget_used = initial_budget - final_budget
        
        # Check if budget usage is justified by outcome quality
        avg_outcome_quality = np.mean([step.outcome_quality for step in self.steps])
        efficiency = avg_outcome_quality / (budget_used + 0.01)  # Avoid division by zero
        
        return efficiency >= 0.5  # Minimum efficiency threshold
    
    def get_trajectory_alignment_metrics(self) -> Dict[str, float]:
        """Get comprehensive alignment metrics for the trajectory"""
        return {
            'cumulative_alignment_score': self.compute_alignment_score(),
            'consistency_maintained': float(self.check_consistency_constraint()),
            'progressive_refinement': float(self.check_progressive_refinement()),
            'resource_rational': float(self.check_resource_rationality()),
            'average_step_alignment': np.mean([step.alignment_score for step in self.steps]) if self.steps else 0.0,
            'trajectory_length': len(self.steps),
            'final_outcome_quality': self.steps[-1].outcome_quality if self.steps else 0.0
        }
    
    def visualize_trajectory(self):
        """Visualize trajectory alignment and quality progression"""
        if not self.steps:
            print("No trajectory data to visualize")
            return
            
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # 1. Alignment scores over time
        alignment_scores = [step.alignment_score for step in self.steps]
        axes[0,0].plot(range(len(alignment_scores)), alignment_scores, 'bo-', linewidth=2)
        axes[0,0].set_title('Alignment Scores Over Time')
        axes[0,0].set_xlabel('Step')
        axes[0,0].set_ylabel('Alignment Score')
        axes[0,0].grid(True, alpha=0.3)
        
        # 2. Outcome quality progression
        outcome_qualities = [step.outcome_quality for step in self.steps]
        axes[0,1].plot(range(len(outcome_qualities)), outcome_qualities, 'go-', linewidth=2)
        axes[0,1].set_title('Outcome Quality Progression')
        axes[0,1].set_xlabel('Step')
        axes[0,1].set_ylabel('Outcome Quality')
        axes[0,1].grid(True, alpha=0.3)
        
        # 3. Tool usage distribution
        tool_counts = defaultdict(int)
        for step in self.steps:
            tool_counts[step.action.name] += 1
            
        tools = list(tool_counts.keys())
        counts = list(tool_counts.values())
        
        axes[1,0].bar(range(len(tools)), counts, alpha=0.7)
        axes[1,0].set_title('Tool Usage Distribution')
        axes[1,0].set_xlabel('Tools')
        axes[1,0].set_ylabel('Usage Count')
        axes[1,0].set_xticks(range(len(tools)))
        axes[1,0].set_xticklabels([tool.replace('_', ' ')[:8] for tool in tools], rotation=45)
        
        # 4. Cumulative rewards
        rewards = [step.reward for step in self.steps]
        cumulative_rewards = np.cumsum(rewards)
        axes[1,1].plot(range(len(cumulative_rewards)), cumulative_rewards, 'ro-', linewidth=2)
        axes[1,1].set_title('Cumulative Rewards')
        axes[1,1].set_xlabel('Step')
        axes[1,1].set_ylabel('Cumulative Reward')
        axes[1,1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Print trajectory metrics
        metrics = self.get_trajectory_alignment_metrics()
        print("\nTrajectory Alignment Metrics:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.3f}")

# Test trajectory system
trajectory = AlignedTrajectory()

# Create sample trajectory steps
for i in range(5):
    # Create evolving state
    problem = ProblemState('analytical', 0.6 + i*0.05, 'science', ['researchers'])
    context = ContextState(0.3 + i*0.1, 0.8, 0.7, 0.4)
    resources = ResourceState(0.8 - i*0.1, 0.9 - i*0.15, 
                             {tool.name: True for tool in ResearchTool}, 
                             {tool.name: 0.9 - i*0.1 for tool in ResearchTool})
    constraints = ConstraintState(0.6, ['ethical'], 
                                 {'accuracy': 0.8, 'speed': 0.4, 'cost': 0.6, 'safety': 0.9},
                                 {'bias': 0.1, 'harm': 0.05})
    
    state = AgentState(problem, context, resources, constraints)
    
    # Select action based on step (simulating learning progression)
    actions = [ResearchTool.WEB_SEARCH, ResearchTool.ACADEMIC_SEARCH, 
              ResearchTool.FACT_CHECK, ResearchTool.CROSS_REFERENCE, ResearchTool.SYNTHESIS]
    action = actions[i]
    
    # Get reward from our reward matrix
    reward = reward_matrix.get_stochastic_reward(state, action)
    
    # Create next state (simplified transition)
    next_state = AgentState(problem, context, resources, constraints)
    
    # Calculate alignment score and outcome quality
    alignment_score = reward_matrix.get_alignment_reward(state, action)
    outcome_quality = 0.5 + i*0.1 + np.random.normal(0, 0.05)  # Progressive improvement
    
    step = TrajectoryStep(
        state=state,
        action=action,
        reward=reward,
        next_state=next_state,
        alignment_score=alignment_score,
        outcome_quality=outcome_quality,
        info={'step': i, 'tool_cost': action_space.get_tool_properties(action).cost}
    )
    
    trajectory.add_step(step)

print(f"Created trajectory with {trajectory.get_trajectory_length()} steps")
print(f"Cumulative alignment score: {trajectory.compute_alignment_score():.3f}")
print(f"Consistency maintained: {trajectory.check_consistency_constraint()}")
print(f"Progressive refinement: {trajectory.check_progressive_refinement()}")
print(f"Resource rational: {trajectory.check_resource_rationality()}")

In [ ]:
# Visualize the trajectory
trajectory.visualize_trajectory()

## Part 2: Curriculum Learning System

Implementing the 4-stage curriculum learning system from module3.md:
1. **Stage 1**: Single-Tool Mastery
2. **Stage 2**: Sequential Decision-Making  
3. **Stage 3**: Stochastic Adaptation
4. **Stage 4**: Adversarial Robustness

With mathematical advancement criteria and transfer learning between stages.

In [ ]:
class CurriculumStage(Enum):
    """Four stages of curriculum learning"""
    SINGLE_TOOL_MASTERY = 1
    SEQUENTIAL_DECISIONS = 2
    STOCHASTIC_ADAPTATION = 3
    ADVERSARIAL_ROBUSTNESS = 4

@dataclass
class StageConfig:
    """Configuration for each curriculum stage"""
    stage: CurriculumStage
    max_complexity: float
    max_tools_per_problem: int
    reward_variance: float
    trajectory_length: int
    advancement_thresholds: Dict[str, float]
    problem_distribution: Dict[str, float]
    
class CurriculumLearningSystem:
    """Complete curriculum learning system with progressive difficulty"""
    
    def __init__(self, action_space: MultiToolActionSpace, reward_matrix: StochasticRewardMatrix):
        self.action_space = action_space
        self.reward_matrix = reward_matrix
        self.current_stage = CurriculumStage.SINGLE_TOOL_MASTERY
        
        # Define stage configurations
        self.stage_configs = {
            CurriculumStage.SINGLE_TOOL_MASTERY: StageConfig(
                stage=CurriculumStage.SINGLE_TOOL_MASTERY,
                max_complexity=0.3,
                max_tools_per_problem=1,
                reward_variance=0.5,
                trajectory_length=1,
                advancement_thresholds={
                    'performance': 7.0,
                    'alignment': 0.8,
                    'variance': 0.4,
                    'robustness': 6.0
                },
                problem_distribution={'factual': 0.6, 'analytical': 0.3, 'urgent': 0.1, 'controversial': 0.0}
            ),
            CurriculumStage.SEQUENTIAL_DECISIONS: StageConfig(
                stage=CurriculumStage.SEQUENTIAL_DECISIONS,
                max_complexity=0.6,
                max_tools_per_problem=3,
                reward_variance=1.0,
                trajectory_length=5,
                advancement_thresholds={
                    'performance': 6.5,
                    'alignment': 0.75,
                    'variance': 0.6,
                    'robustness': 5.5
                },
                problem_distribution={'factual': 0.4, 'analytical': 0.4, 'urgent': 0.15, 'controversial': 0.05}
            ),
            CurriculumStage.STOCHASTIC_ADAPTATION: StageConfig(
                stage=CurriculumStage.STOCHASTIC_ADAPTATION,
                max_complexity=0.8,
                max_tools_per_problem=6,
                reward_variance=2.0,
                trajectory_length=8,
                advancement_thresholds={
                    'performance': 6.0,
                    'alignment': 0.7,
                    'variance': 0.8,
                    'robustness': 5.0
                },
                problem_distribution={'factual': 0.25, 'analytical': 0.35, 'urgent': 0.25, 'controversial': 0.15}
            ),
            CurriculumStage.ADVERSARIAL_ROBUSTNESS: StageConfig(
                stage=CurriculumStage.ADVERSARIAL_ROBUSTNESS,
                max_complexity=1.0,
                max_tools_per_problem=12,
                reward_variance=3.0,
                trajectory_length=12,
                advancement_thresholds={
                    'performance': 5.5,
                    'alignment': 0.65,
                    'variance': 1.0,
                    'robustness': 4.5
                },
                problem_distribution={'factual': 0.2, 'analytical': 0.3, 'urgent': 0.3, 'controversial': 0.2}
            )
        }
        
        # Learning history and statistics
        self.stage_history = []
        self.performance_history = defaultdict(list)
        self.q_values = defaultdict(lambda: defaultdict(float))  # Q(s,a) values
        
    def generate_problem_for_stage(self, stage: CurriculumStage) -> AgentState:
        """Generate a problem appropriate for the current stage"""
        config = self.stage_configs[stage]
        
        # Sample problem type from stage distribution
        problem_types = list(config.problem_distribution.keys())
        problem_weights = list(config.problem_distribution.values())
        problem_type = np.random.choice(problem_types, p=problem_weights)
        
        # Generate complexity within stage limits
        complexity = np.random.uniform(0.1, config.max_complexity)
        
        # Create problem state
        problem = ProblemState(
            query_type=problem_type,
            complexity_level=complexity,
            domain=np.random.choice(['science', 'politics', 'technology', 'general']),
            stakeholders=['user'] + [f'stakeholder_{i}' for i in range(np.random.randint(0, 3))]
        )
        
        # Create appropriate context based on stage
        if stage == CurriculumStage.SINGLE_TOOL_MASTERY:
            # Low pressure, high quality requirements
            context = ContextState(0.2, 0.9, 0.8, 0.3)
        elif stage == CurriculumStage.SEQUENTIAL_DECISIONS:
            # Moderate pressure, moderate quality
            context = ContextState(0.4, 0.7, 0.6, 0.5)
        elif stage == CurriculumStage.STOCHASTIC_ADAPTATION:
            # High variability in context
            context = ContextState(np.random.uniform(0.1, 0.8), np.random.uniform(0.5, 0.9),
                                 np.random.uniform(0.4, 0.8), np.random.uniform(0.2, 0.8))
        else:  # ADVERSARIAL_ROBUSTNESS
            # Extreme and adversarial contexts
            context = ContextState(np.random.uniform(0.6, 1.0), np.random.uniform(0.8, 1.0),
                                 np.random.uniform(0.2, 0.6), np.random.uniform(0.7, 1.0))
        
        # Create resources and constraints
        resources = ResourceState(
            budget_remaining=np.random.uniform(0.5, 1.0),
            time_remaining=np.random.uniform(0.3, 1.0),
            tool_availability={tool.name: True for tool in ResearchTool},
            api_limits={tool.name: np.random.uniform(0.5, 1.0) for tool in ResearchTool}
        )
        
        # Vary user values by stage
        if stage == CurriculumStage.SINGLE_TOOL_MASTERY:
            # Clear value priorities
            user_values = {'accuracy': 0.9, 'speed': 0.3, 'cost': 0.5, 'safety': 0.8}
        else:
            # More complex value trade-offs in later stages
            user_values = {
                'accuracy': np.random.uniform(0.6, 0.95),
                'speed': np.random.uniform(0.2, 0.8),
                'cost': np.random.uniform(0.3, 0.8),
                'safety': np.random.uniform(0.7, 0.95)
            }
        
        constraints = ConstraintState(
            privacy_level=np.random.uniform(0.3, 0.8),
            compliance_requirements=['ethical'] if stage.value >= 2 else [],
            user_values=user_values,
            safety_thresholds={'bias': 0.1, 'harm': 0.05}
        )
        
        return AgentState(problem, context, resources, constraints)
    
    def restrict_actions_for_stage(self, stage: CurriculumStage, state: AgentState) -> List[ResearchTool]:
        """Restrict available actions based on stage"""
        config = self.stage_configs[stage]
        all_tools = list(ResearchTool)
        
        if stage == CurriculumStage.SINGLE_TOOL_MASTERY:
            # Only allow single best tool for each problem type
            if state.problem.query_type == 'factual':
                return [ResearchTool.ACADEMIC_SEARCH, ResearchTool.FACT_CHECK]
            elif state.problem.query_type == 'urgent':
                return [ResearchTool.WEB_SEARCH, ResearchTool.NEWS_SEARCH]
            else:
                return [ResearchTool.ACADEMIC_SEARCH, ResearchTool.WEB_SEARCH]
        
        elif stage == CurriculumStage.SEQUENTIAL_DECISIONS:
            # Allow information gathering + analysis tools
            return [ResearchTool.ACADEMIC_SEARCH, ResearchTool.WEB_SEARCH, 
                   ResearchTool.FACT_CHECK, ResearchTool.SUMMARIZATION,
                   ResearchTool.CONFIDENCE_ASSESSMENT, ResearchTool.SYNTHESIS]
        
        elif stage == CurriculumStage.STOCHASTIC_ADAPTATION:
            # Allow most tools except human consultation
            return [tool for tool in all_tools if tool != ResearchTool.HUMAN_CONSULTATION]
        
        else:  # ADVERSARIAL_ROBUSTNESS
            # All tools available
            return all_tools
    
    def evaluate_stage_performance(self, stage: CurriculumStage, trajectories: List[AlignedTrajectory]) -> Dict[str, float]:
        """Evaluate performance metrics for stage advancement"""
        if not trajectories:
            return {'performance': 0, 'alignment': 0, 'variance': float('inf'), 'robustness': 0}
        
        # Calculate performance metrics
        rewards = []
        alignment_scores = []
        final_qualities = []
        
        for traj in trajectories:
            if traj.steps:
                traj_rewards = [step.reward for step in traj.steps]
                rewards.extend(traj_rewards)
                
                alignment_scores.append(traj.compute_alignment_score())
                final_qualities.append(traj.steps[-1].outcome_quality)
        
        if not rewards:
            return {'performance': 0, 'alignment': 0, 'variance': float('inf'), 'robustness': 0}
        
        performance = np.mean(rewards)
        alignment = np.mean(alignment_scores) if alignment_scores else 0
        variance = np.std(rewards)
        robustness = np.min(rewards) if rewards else 0  # Worst-case performance
        
        return {
            'performance': performance,
            'alignment': alignment,
            'variance': variance,
            'robustness': robustness
        }
    
    def check_advancement_criteria(self, stage: CurriculumStage, metrics: Dict[str, float]) -> bool:
        """Check if agent meets criteria to advance to next stage"""
        config = self.stage_configs[stage]
        thresholds = config.advancement_thresholds
        
        performance_met = metrics['performance'] >= thresholds['performance']
        alignment_met = metrics['alignment'] >= thresholds['alignment']
        variance_ok = metrics['variance'] <= thresholds['variance']
        robustness_met = metrics['robustness'] >= thresholds['robustness']
        
        return performance_met and alignment_met and variance_ok and robustness_met
    
    def transfer_knowledge(self, from_stage: CurriculumStage, to_stage: CurriculumStage):
        """Transfer learned Q-values between stages"""
        # Simplified transfer: copy Q-values for overlapping state-action pairs
        # In practice, this would use more sophisticated transfer learning
        print(f"Transferring knowledge from {from_stage.name} to {to_stage.name}")
        
        # Transfer would involve mapping state representations and updating Q-values
        # For now, we'll implement a simplified version
        transfer_efficiency = 0.8  # How much knowledge transfers
        
        # This is a placeholder for actual transfer learning implementation
        print(f"Transfer efficiency: {transfer_efficiency:.2f}")
    
    def run_curriculum_stage(self, stage: CurriculumStage, num_episodes: int = 100) -> Dict[str, float]:
        """Run training for a specific curriculum stage"""
        print(f"\n=== Running {stage.name} Stage ===")
        print(f"Episodes: {num_episodes}")
        
        trajectories = []
        config = self.stage_configs[stage]
        
        for episode in range(num_episodes):
            # Generate problem for this stage
            state = self.generate_problem_for_stage(stage)
            available_actions = self.restrict_actions_for_stage(stage, state)
            
            # Create trajectory
            trajectory = AlignedTrajectory(max_length=config.trajectory_length)
            
            # Simulate trajectory (simplified RL)
            current_state = state
            for step in range(config.trajectory_length):
                # Epsilon-greedy action selection (simplified)
                epsilon = 0.1
                if np.random.random() < epsilon:
                    action = np.random.choice(available_actions)
                else:
                    # Select action with highest Q-value
                    state_key = str(current_state.to_vector()[:5])  # Simplified state key
                    q_values = {a: self.q_values[state_key][a] for a in available_actions}
                    action = max(q_values, key=q_values.get) if q_values else np.random.choice(available_actions)
                
                # Get reward with stage-appropriate variance
                base_reward = self.reward_matrix.get_stochastic_reward(current_state, action)
                stage_noise = np.random.normal(0, config.reward_variance)
                reward = base_reward + stage_noise
                
                # Calculate alignment score
                alignment_score = self.reward_matrix.get_alignment_reward(current_state, action)
                
                # Create next state (simplified transition)
                next_state = self.generate_problem_for_stage(stage)
                
                # Outcome quality improves with experience
                outcome_quality = min(1.0, 0.3 + (episode / num_episodes) * 0.6 + np.random.normal(0, 0.1))
                
                # Create trajectory step
                traj_step = TrajectoryStep(
                    state=current_state,
                    action=action,
                    reward=reward,
                    next_state=next_state,
                    alignment_score=alignment_score,
                    outcome_quality=outcome_quality,
                    info={'episode': episode, 'step': step, 'stage': stage.name}
                )
                
                trajectory.add_step(traj_step)
                
                # Update Q-values (simplified Q-learning)
                state_key = str(current_state.to_vector()[:5])
                next_state_key = str(next_state.to_vector()[:5])
                
                # Q-learning update
                learning_rate = 0.1
                discount = 0.95
                
                max_next_q = max([self.q_values[next_state_key][a] for a in available_actions], default=0)
                self.q_values[state_key][action] += learning_rate * (
                    reward + discount * max_next_q - self.q_values[state_key][action]
                )
                
                current_state = next_state
                
                # Break if this is single-step stage
                if config.trajectory_length == 1:
                    break
            
            trajectories.append(trajectory)
            
            # Progress update
            if (episode + 1) % 20 == 0:
                recent_rewards = [sum(step.reward for step in traj.steps) 
                                for traj in trajectories[-20:]]
                avg_reward = np.mean(recent_rewards)
                print(f"Episode {episode + 1}: Avg Reward = {avg_reward:.2f}")
        
        # Evaluate final performance
        metrics = self.evaluate_stage_performance(stage, trajectories)
        print(f"\nStage {stage.name} Results:")
        for metric, value in metrics.items():
            print(f"  {metric}: {value:.3f}")
        
        # Store performance history
        self.performance_history[stage] = trajectories
        
        return metrics
    
    def visualize_curriculum_progress(self, results: Dict[CurriculumStage, Dict[str, float]]):
        """Visualize learning progress across curriculum stages"""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        stages = list(results.keys())
        stage_names = [stage.name.replace('_', ' ') for stage in stages]
        
        # 1. Performance progression
        performances = [results[stage]['performance'] for stage in stages]
        axes[0,0].plot(range(len(stages)), performances, 'bo-', linewidth=2, markersize=8)
        axes[0,0].set_title('Performance Across Curriculum Stages')
        axes[0,0].set_xlabel('Stage')
        axes[0,0].set_ylabel('Average Performance')
        axes[0,0].set_xticks(range(len(stages)))
        axes[0,0].set_xticklabels(stage_names, rotation=45)
        axes[0,0].grid(True, alpha=0.3)
        
        # 2. Alignment scores
        alignments = [results[stage]['alignment'] for stage in stages]
        axes[0,1].plot(range(len(stages)), alignments, 'go-', linewidth=2, markersize=8)
        axes[0,1].set_title('Alignment Scores Across Stages')
        axes[0,1].set_xlabel('Stage')
        axes[0,1].set_ylabel('Alignment Score')
        axes[0,1].set_xticks(range(len(stages)))
        axes[0,1].set_xticklabels(stage_names, rotation=45)
        axes[0,1].grid(True, alpha=0.3)
        
        # 3. Variance and robustness
        variances = [results[stage]['variance'] for stage in stages]
        robustness = [results[stage]['robustness'] for stage in stages]
        
        x = np.arange(len(stages))
        width = 0.35
        
        axes[1,0].bar(x - width/2, variances, width, label='Variance', alpha=0.7)
        axes[1,0].bar(x + width/2, robustness, width, label='Robustness', alpha=0.7)
        axes[1,0].set_title('Variance vs Robustness')
        axes[1,0].set_xlabel('Stage')
        axes[1,0].set_ylabel('Value')
        axes[1,0].set_xticks(range(len(stages)))
        axes[1,0].set_xticklabels(stage_names, rotation=45)
        axes[1,0].legend()
        axes[1,0].grid(True, alpha=0.3)
        
        # 4. Advancement criteria heatmap
        metrics = ['performance', 'alignment', 'variance', 'robustness']
        advancement_data = np.array([[results[stage][metric] for metric in metrics] 
                                   for stage in stages])
        
        sns.heatmap(advancement_data, 
                   xticklabels=metrics,
                   yticklabels=stage_names,
                   annot=True, fmt='.2f', 
                   ax=axes[1,1])
        axes[1,1].set_title('All Metrics Across Stages')
        
        plt.tight_layout()
        plt.show()

# Test the curriculum learning system
curriculum = CurriculumLearningSystem(action_space, reward_matrix)

# Show stage configurations
print("Curriculum Stage Configurations:")
print("=" * 40)
for stage, config in curriculum.stage_configs.items():
    print(f"\n{stage.name}:")
    print(f"  Max Complexity: {config.max_complexity}")
    print(f"  Max Tools: {config.max_tools_per_problem}")
    print(f"  Trajectory Length: {config.trajectory_length}")
    print(f"  Advancement Thresholds: {config.advancement_thresholds}")
    print(f"  Problem Distribution: {config.problem_distribution}")

# Test stage generation
print("\nSample problems for each stage:")
for stage in CurriculumStage:
    problem = curriculum.generate_problem_for_stage(stage)
    actions = curriculum.restrict_actions_for_stage(stage, problem)
    print(f"\n{stage.name}:")
    print(f"  Problem: {problem.problem.query_type}, complexity={problem.problem.complexity_level:.2f}")
    print(f"  Available actions: {len(actions)} tools")

## Part 3: Running and Evaluating Curriculum Stages

In [ ]:
# Quick demonstration: Run a single curriculum stage
print("\n" + "="*60)
print("DEMONSTRATION: Running Single Curriculum Stage")
print("="*60)

# Run just the first stage as demonstration
stage1_results = curriculum.run_curriculum_stage(CurriculumStage.SINGLE_TOOL_MASTERY, num_episodes=50)

print("\nStage 1 completed! Check if advancement criteria are met:")
can_advance = curriculum.check_advancement_criteria(CurriculumStage.SINGLE_TOOL_MASTERY, stage1_results)
print(f"Ready to advance to Stage 2: {can_advance}")

In [ ]:
# Optional: Run multiple stages and visualize progression
print("\n" + "="*60)
print("FULL CURRICULUM DEMONSTRATION")
print("="*60)

# Run first two stages
results = {}
stages_to_run = [CurriculumStage.SINGLE_TOOL_MASTERY, CurriculumStage.SEQUENTIAL_DECISIONS]

for stage in stages_to_run:
    print(f"\n{'='*50}")
    print(f"Running {stage.name}")
    print(f"{'='*50}")
    
    stage_results = curriculum.run_curriculum_stage(stage, num_episodes=30)
    results[stage] = stage_results
    
    # Check advancement
    can_advance = curriculum.check_advancement_criteria(stage, stage_results)
    print(f"\nAdvancement Check for {stage.name}:")
    config = curriculum.stage_configs[stage]
    for metric, threshold in config.advancement_thresholds.items():
        actual = stage_results[metric]
        status = "✓" if (actual >= threshold if metric != 'variance' else actual <= threshold) else "✗"
        print(f"  {metric}: {actual:.3f} {'≥' if metric != 'variance' else '≤'} {threshold:.3f} {status}")
    
    print(f"Can advance: {'Yes' if can_advance else 'No'}")
    
    if can_advance and stage != stages_to_run[-1]:
        next_stage = CurriculumStage(stage.value + 1)
        curriculum.transfer_knowledge(stage, next_stage)

print(f"\n{'='*60}")
print("CURRICULUM RESULTS SUMMARY")
print(f"{'='*60}")

for stage, metrics in results.items():
    print(f"\n{stage.name}:")
    for metric, value in metrics.items():
        print(f"  {metric}: {value:.3f}")

# Visualize results if we have multiple stages
if len(results) > 1:
    curriculum.visualize_curriculum_progress(results)

## Key Learning Outcomes - Part 2

You've successfully implemented trajectory-level alignment and curriculum learning:

### 1. **Trajectory-Level Alignment**
- ✅ **Sequential Constraint Checking**: Alignment verified at each step, not just endpoints
- ✅ **Consistency Constraints**: Similar states lead to value-consistent actions
- ✅ **Progressive Refinement**: Information quality improves throughout trajectories
- ✅ **Resource Rationality**: Efficient use of computational resources

### 2. **Curriculum Learning System**
- ✅ **4-Stage Progression**: Single-tool → Sequential → Stochastic → Adversarial
- ✅ **Mathematical Advancement Criteria**: Formal thresholds for stage progression
- ✅ **Progressive Complexity**: Gradual increase in problem difficulty and tool availability
- ✅ **Transfer Learning**: Knowledge transfer between curriculum stages

### 3. **Alignment Preservation**
- ✅ **Stage-Specific Constraints**: Each stage maintains alignment requirements
- ✅ **Cumulative Alignment Scoring**: Weighted sum across trajectory steps
- ✅ **Value Consistency**: User preferences respected throughout learning
- ✅ **Robustness Measures**: Worst-case performance tracking

## Mathematical Insights

**Trajectory Alignment Formula:**
$$A_{\text{cumulative}}(\tau) = \sum_{t=0}^T \alpha^t \cdot \text{alignment\_score}(s_t, a_t)$$

**Advancement Criteria:**
- Performance: $\mathbb{E}[R(s,a)] \geq \tau_{\text{perf}}$
- Alignment: $\mathbb{E}[A(s,a)] \geq \tau_{\text{align}}$
- Variance: $\text{Var}[R(s,a)] \leq \sigma_{\text{thresh}}$
- Robustness: $\min[R(s,a)] \geq \tau_{\text{robust}}$

## Next Steps

**Continue to Part 3** to see how these components integrate into:
- Complete simulation environment
- Real-world experimentation
- Claude API integration
- Hands-on exercises and analysis

**Advanced Experiments:**
1. Modify advancement thresholds and observe learning progression
2. Test different trajectory lengths and complexity curves
3. Implement more sophisticated transfer learning mechanisms
4. Design custom curriculum stages for specific domains

The trajectory and curriculum systems you've built provide the scaffolding for safe, aligned learning in complex multi-tool environments!